In [32]:
import pandas as pd
import os
import weaviate
import weaviate.classes.config as wvc
import weaviate.classes.data as wvd
import weaviate.classes.query as wvq
from weaviate.util import generate_uuid5

from dotenv import load_dotenv
from openai import OpenAI

In [4]:
client = weaviate.connect_to_local(
    headers={"X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")}
)
client.connect()

### Introduction

The system designing process including:
1. storing the knowledge graph into vector databases: weaviate
2. traverse the knowledge graph to get most relevant context for answering the question
3. final answer generation

### Step 1: Store the knowledge graph into a vector database: weaviate

#### Step 1.1 Define schema for vector database

Two vector collections: 
1. the entity collection
2. the links (edges/relaitonships) collection

To make the traverse convenient:
1. Each entity will link to links (with link ids)
2. Each link link to other links (linka: entityA -> entityB, linkb: entityB -> entityC; then linka link to linkb)
3. All these links are directed.

In [ ]:
# client.collections.delete("KGLinks")
# client.collections.delete("KGEntities")

if not client.collections.exists("KGLinks"):
    print("not exists, create collection")
    client.collections.create(
        name="KGLinks",
        description="A unified collection for wildfire relationships",
        vector_config=[wvc.Configure.Vectors.text2vec_openai(
                name='core',
                source_properties=["link_name", "source_entity", "target_entity"],
                model="text-embedding-3-large",
                dimensions=1024
            )], 
        properties=[
            # 1. 核心分类属性：用于“只检索 FireEvent 类”
            wvc.Property(name="link_name", data_type=wvc.DataType.TEXT), 
            wvc.Property(name="source_entity", data_type=wvc.DataType.TEXT), 
            wvc.Property(name="target_entity", data_type=wvc.DataType.TEXT),
            wvc.Property(name="link_description", data_type=wvc.DataType.TEXT)
        ],
        # 4. 自引用（Self-Reference）：实现“链接到关联实体”
        references=[
            wvc.ReferenceProperty(
                name="links",
                target_collection="KGLinks", # 指向自己这个 Collection
                description="Directed links to downstream links"
            )
        ]
    )

if not client.collections.exists("KGEntities"):
    print("not exists, create collection")
    client.collections.create(
        name="KGEntities",
        description="A unified collection for wildfire entities and their relationships",
        vector_config=[wvc.Configure.Vectors.text2vec_openai(
                name='core',
                source_properties=["entity_name", "entity_type"],
                model="text-embedding-3-large",
                dimensions=1024
            )], 
        properties=[
            # 1. 核心分类属性：用于“只检索 FireEvent 类”
            wvc.Property(name="entity_name", data_type=wvc.DataType.TEXT), 
            wvc.Property(name="entity_type", data_type=wvc.DataType.TEXT), 
            wvc.Property(name="entity_description", data_type=wvc.DataType.TEXT)
        ],
        # 4. 自引用（Self-Reference）：实现“链接到关联实体”
        references=[
            wvc.ReferenceProperty(
                name="links",
                target_collection="KGLinks", # 指向自己这个 Collection
                description="Links to related KG links"
            )
        ]
    )

not exists, create collection
not exists, create collection


In [ ]:
entity_collection = client.collections.use("KGEntities")
link_collection = client.collections.use("KGLinks")

#### Step 1.2 vectorize entities and links into corresponding vector database collections

In [ ]:
# vectorize entities
entities = pd.read_csv(r"merged_entities.csv")

entity_objects = []
for i in range(entities.shape[0]):
    entity = entities.iloc[i]
    entity_objects.append(
        wvd.DataObject(
            properties={"entity_name": entity["entity_name"],
                        "entity_type": entity["entity_type"],
                        "entity_description": entity["entity_description"]},
            uuid=generate_uuid5(entity["entity_name"])
        )
    )

entity_collection.data.insert_many(entity_objects)

BatchObjectReturn(_all_responses=[UUID('9220dfee-5c37-505d-a4c5-4fe9a28a9b17'), UUID('f2bca0bf-3cec-5c85-8814-e8a43db4fb63'), UUID('e7e3eb79-3480-50e6-9c91-673dba209143'), UUID('018561ec-7c56-5b50-b051-0ea3c48af5fb'), UUID('33579f9a-a0d2-52fd-9282-e71fab879438'), UUID('8044e6ed-1ebb-58fd-bd2a-dc551840fc89'), UUID('d07af5ac-4f0d-5af1-9ded-427d03b58075'), UUID('7f4546b2-d090-5cb4-bc2e-c4046b1f3514'), UUID('c30e16c0-82a4-5ef4-b159-b22896830445'), UUID('4d84249b-4f85-565b-a7d3-ab6f93bac0bf'), UUID('20614fff-fd58-5ef3-8b39-37c46204b2aa'), UUID('d3d6e19e-4b2d-5d98-98b7-c1b02ee50f5d'), UUID('6a28e345-203b-5c85-8ac0-3d66c2dba290'), UUID('1c87f928-fd2c-55c7-9cf3-ff30c95becb5'), UUID('b86f68ad-6497-5350-ba24-bddee28a6d78'), UUID('8e122c88-674e-5a81-aecc-b7fbf7166f1a'), UUID('367148a6-c6ac-5489-96c7-67af1da49f53'), UUID('0f7fb9b8-897b-59f7-8e2c-985d0cab4a04'), UUID('7ac30fed-0458-5d26-aa2f-056ecf3eb1ec'), UUID('0f54d063-0e6e-512a-bf67-a859216be27d'), UUID('7bec0c94-7f9c-5c3f-bf03-4897c7d8b09e'), 

In [ ]:
# vectorize links
links = pd.read_csv(r"merged_relationships.csv")

links_objects = []
for i in range(links.shape[0]):
    link = links.iloc[i]
    # print(link)
    try:
        source_entity = link["source"]
        target_entity = link["target"]
        source_obj = entity_collection.query.fetch_object_by_id(
            uuid=generate_uuid5(source_entity)
            )
        source_entity_description = f'Entity Name: {source_entity}. \n Entity Type: {source_obj.properties["entity_type"]}. \n Entity Description: {source_obj.properties["entity_description"]}'
    except:   
        source_entity_description = ""
    
    try:
        target_obj = entity_collection.query.fetch_object_by_id(
            uuid=generate_uuid5(target_entity)
            )
        target_entity_description = f'Entity Name: {target_entity}. \n Entity Type: {target_obj.properties["entity_type"]}. \n Entity Description: {target_obj.properties["entity_description"]}'
    except:
        target_entity_description = ""    

    link_description = f'Link Name: {link["relationship"]}. \n Source Entity Description: {source_entity_description}. \n Target Entity Description: {target_entity_description}. \n Link Description: {link["relationship_description"]}'
    
    links_objects.append(
        wvd.DataObject(
            properties={"link_name": link["relationship"],
                        "source_entity": link["source"],
                        "target_entity": link["target"],
                        "link_description": link_description},
            uuid=generate_uuid5(f'{link["source"]} {link["target"]} {link["relationship"]}')
        )
    )

link_collection.data.insert_many(links_objects)

BatchObjectReturn(_all_responses=[UUID('2cff86f3-b2c9-5700-b441-768d270ddd33'), UUID('e4afa70f-2043-5762-ba18-206e0e46ba8d'), UUID('0e30f43e-52bc-59e5-96e3-254f49df8cb2'), UUID('9ae5e3b4-5945-505e-bca2-3b8da2a79ea1'), UUID('6de079a7-2248-50fb-8cdc-f0f145f554c5'), UUID('0ff5c078-9d6b-5048-ba41-7b4c3a2549f3'), UUID('bbd09a5a-4d1f-5c04-8021-a828fd542b4f'), UUID('05d45e10-b830-547e-868e-b155268edbf3'), UUID('f6d31cea-b0ad-5222-899c-7394b717c9da'), UUID('3cdf357d-8013-5786-9734-4eef6477209e'), UUID('cbadb4a2-d300-5766-bab2-1a2742b2b77d'), UUID('12086a52-a169-5ca9-9e80-d80afeae7e59'), UUID('62cc0b9b-d3c2-5e52-8da0-868bee0a6be3'), UUID('5b28bd75-0834-5d4f-b3fc-b01b40d7740f'), UUID('3749e95e-27b4-574d-a7aa-0b63765ffd87'), UUID('6bed34c5-c7a9-51fe-9cb9-09badb4821bf'), UUID('75032cc9-7ff3-5a64-9729-c1fcd1998b9e'), UUID('239605ec-50d6-5d92-89b9-232d101827ef'), UUID('4d35ee9c-3e18-5d2d-8bdf-0b614b3c8b04'), UUID('a9e53838-66f9-5e8f-a2ee-bd8a4e24add0'), UUID('94eaddfe-b889-524b-80d5-25f860b850d7'), 

#### Step 1.3 add references, so entities can reference to links, and links and reference to downstream links

In [ ]:
# class to links
for i in range(entities.shape[0]):

    row = entities.iloc[i]
    entitiy_id = generate_uuid5(row['entity_name'])

    sub_link = links[links['source'] == row['entity_name']]
    if sub_link.shape[0] == 0:
        continue
    else:
        try:
            for j in range(sub_link.shape[0]):
                l = sub_link.iloc[j]
                link_id = generate_uuid5(f'{l["source"]} {l["target"]} {l["relationship"]}')
                entity_collection.data.reference_add(
                    from_uuid=entitiy_id,
                    from_property="links",
                    to=link_id
                )
        except:
            print(row['entity_name'])
            print(f'{l["source"]} {l["target"]} {l["relationship"]}')
            continue

In [91]:
# link to links
for i in range(links.shape[0]):
    row = links.iloc[i]
    link1_id = generate_uuid5(f'{row["source"]} {row["target"]} {row["relationship"]}')
    target = row['target']

    sub_link = links[links['source'] == target]
    if sub_link.shape[0] == 0:
        continue
    else:
        for j in range(sub_link.shape[0]):
            l = sub_link.iloc[j]
            link2_id = generate_uuid5(f'{l["source"]} {l["target"]} {l["relationship"]}')
            link_collection.data.reference_add(
                from_uuid=link1_id,
                from_property='links',
                to=link2_id
            )

### Step 2 QA: Graph Traverse

The question answering process consists of three steps:
1. question decomposing: this will use LLMs to decompose the question to extract the entities and corresponding entity types for answering the question.
2. Graph traversing: The extracted entity information from step 1 will be used as the start point for the graph traverse process. Then for each iteration, the system will expore all linked links to rich the context. Then a LLM based context evaluator will evaluate the existing context: if it is good to answer the question, stop iteration and generate the answer; if not good enough, keep traversing the graph unitl stop criteria (5 iterations) or get good context.
3. answer generation based on the extracted context and the question.

#### Step 2.1 LLM templates

In [ ]:
# template for question decomposition
Traverse_Start_Point_Template = """
    Analysis the question and infer the entity and corresponding entity type knowledge graph that are necessary to answer this.

    A class in a knowledge graph ontology defines the entities types.

    These classes information should include the potential class names and two or three alternative class names for that class.

    Output two most relevant entities and corresponding types at most.

    ########################################
    entities definition:
    ########################################

    - Fire_Event: The primary wildfire event. This is the root node, all other netities should ideally link back to this event.

    - Fire_Location: Specific geographic areas or neighborhoods.

    - Fire_Cause: The stated reason for the fire's start.

    - Status: This is the Status update entity of a fire event, all the status names should contain the time stamp, e.g., Status_June27_1430. Each status contains (all or part of) the following information:
        - Time_Stamp: The timestamp, all the status should contain the timeline information, e.g., ("June 27", "2:30 PM").
        - Burned_Area: The extent of land damaged (acres/sq miles) at a point in time.
        - Evacuation_Detail: Relocation orders and the scale (number of homes/people) at a point in time.
        - Containment_Status: A specific status of the fire's containment at a point in time.
        - Emergency_Response: Firefighting assets and agency measures at a point in time.
        - Public_Sentiment: Community reactions and emotional tone at a point in time.

    ##################################################
    Example
    ##################################################
    Example 1:
    What is the cause of the XXX Fire?
    Output:
    ("entity":XXX Fire; "entity type": FIRE_EVENT)

    ################################################
    -Real Data-
    #################################################
    Question: {question}
    #################################################
    Output:
"""

# template for context evaluation 
Information_Evaluation_Template = """
Your task is to evaluate if existing information good enough to answer the question.

Requirements for the output:
- Only output "TRUE“ or "FALSE"
- Don't include your analysis process

##############################
Examples
##############################
Example1:
Question: What is the cause of the XXX Fire?
Existing information: The XXX Fire happend in 2021.
Output:
"FALSE"

Example2:
Question: When doese the XXX Fire happened?
Existing information: The XXX Fire happend in the morning on July 1st, 2025.
Output:
"TRUE"

#############################
Real Data
#############################
This is the question: {question}
This is the existing information: {existing_information}
#############################
Output:
"""

# template for answer generation
Question_Answering_Template = """
Answer the question based on the given context information.
This is the question: {question}
This is the existing information: {context}
"""

#### Step 2.2 Graph Traverse

##### Step 2.2.1 Defined functions

In [219]:
# get the start entity and linked links
def parse_start_info(start_entity):
    linked_ids = []
    existing_info = ''
    for obj in start_entity.objects:
        existing_info += str(obj.properties)
        existing_info += '\n'
        existing_info += "##################################"
        existing_info += '\n'
        try:
            linked_ids += [r.uuid for r in obj.references['links'].objects]
        except:
            continue
    return existing_info, linked_ids

# get the linked links of the source link
def parse_linked_links(question, id_list):
    linked_ids = []
    existing_info = ''

    response = link_collection.query.near_text(
        query=question,
        filters=wvq.Filter.by_id().contains_any(id_list),
        limit=3,
        return_metadata=wvq.MetadataQuery(distance=True),
        return_references=[
            wvq.QueryReference(
                link_on='links'
            )])
    for obj in response.objects:
        existing_info += obj.properties['link_description']
        existing_info += '\n'
        existing_info += "##################################"
        existing_info += '\n'
        try:
            linked_ids += [r.uuid for r in obj.references['links'].objects]
        except:
            continue
    return existing_info, linked_ids

# traverse the knowledge graph
def traverse_graph(question, start_entity, LLM_agent, LLM_model_name, max_iteration = 5):
    existing_info, linked_ids = parse_start_info(start_entity)
    for i in range(max_iteration):
        print(str(i))
        existing_info_links, linked_ids = parse_linked_links(question, linked_ids)
        existing_info += existing_info_links
        
        Information_Evaluation_Prompt = Information_Evaluation_Template.format(question=question, existing_information=existing_info)
        evaluation_response = LLM_agent.responses.create(
            model=LLM_model_name,
            input=Information_Evaluation_Prompt
        )
        if 'TRUE' in evaluation_response.output_text:
            print("all information included")
            break
    return existing_info

# the question answering system
def Q_A(question, LLM_agent, LLM_model_name):
    # Question decomposition: get the traverse start point estimation:
    Traverse_Start_Point_Prompt = Traverse_Start_Point_Template.format(question=question)
    start_point_response = LLM_agent.responses.create(
        model=LLM_model_name,
        input=Traverse_Start_Point_Prompt
    )
    start_point_response = start_point_response.output_text

    # Get the start point entity from the knowledge graph based on the start point estimation
    start_entity = entity_collection.query.near_text(
    query=start_point_response,
    limit=1,
    return_metadata=wvq.MetadataQuery(distance=True),
    return_references=[
        wvq.QueryReference(
            link_on='links'
        )]
    )

    # Traverse the knoweldge graph until get all necessary information
    context = traverse_graph(question, start_entity, LLM_agent, LLM_model_name)

    # Answer question based on retrieved context information from knowledge graph
    Question_Answering_Prompt = Question_Answering_Template.format(question=question, context=context)
    final_answer_response = LLM_agent.responses.create(
        model=LLM_model_name,
        input=Question_Answering_Prompt
    )
    
    return final_answer_response.output_text, context

### Answer the question: From July 1 to July 4, how much containment was achieved in the Boulder View Fire?

In [220]:
# Define the agents
LLM_client = OpenAI()
model_name = 'gpt-4.1-mini'

# Question
Question = "From July 1 to July 4, how much containment was achieved in the Boulder View Fire?"

# Answer the question
answer, context = Q_A(Question, LLM_client, model_name)

0
all information included


In [221]:
print(answer)

From July 1 to July 4, the containment of the Boulder View Fire increased from 63% to 95%. Therefore, the containment achieved during this period was 95% - 63% = 32%.


#### Check the context to verify the answer

In [222]:
print(context)

{'entity_description': 'A wildfire event first sparked on June 27, 2024, northwest of Scottsdale that burned 3,711 acres and was 95% contained by July 4, 2024)\n', 'entity_type': 'FIRE_EVENT', 'entity_name': 'BOULDER VIEW FIRE'}
##################################
Link Name: has_status. 
 Source Entity Description: Entity Name: BOULDER VIEW FIRE. 
 Entity Type: FIRE_EVENT. 
 Entity Description: A wildfire event first sparked on June 27, 2024, northwest of Scottsdale that burned 3,711 acres and was 95% contained by July 4, 2024)
. 
 Target Entity Description: Entity Name: STATUS_JULY4. 
 Entity Type: STATUS. 
 Entity Description: [July 4, 2024] Update reporting 95% containment, no recent growth in acreage, and ongoing firefighting efforts)
. 
 Link Description: The July 4 update provides current fire containment and acreage status
##################################
Link Name: caused_by. 
 Source Entity Description: Entity Name: BOULDER VIEW FIRE. 
 Entity Type: FIRE_EVENT. 
 Entity Descr